In [26]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

# Setup Environment
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o", streaming=True)

# Create System Instruction
'''system_message = SystemMessage(
 content=("You are a travel planner assistant."
          "You should only answer travel related questions."
          "If the user asks any question unrelated to travel,respond with 'I can't help with it'."
          "Be concise,helpful and provide accurate information related to travel.")

  )
'''
system_prompt = """
SYSTEM INSTRUCTIONS:
You are a travel planner assistant.
You should only answer travel related questions.
If the user asks any question unrelated to travel,respond with 'I can't help with it'.
Be concise,helpful and provide accurate information related to travel.

{history}

Human: {input}
AI:"""

travel_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=system_prompt
)


class TravelAssistantBot:
    def __init__(self):
        self.history_text = []
        self.msg_count = 0

    def summarize_history(self):
        # Join the list into a block of text for the summarizer to read
        full_text = "\n".join(self.history_text)
        summary_query = f"Summarize these travel plans into one short paragraph:\n{full_text}"

        summary_res = llm.invoke(summary_query)
        summary_text = summary_res.content

        # Clear the list and replace with the summary
        self.history_list = [f"SUMMARY OF PREVIOUS CHAT: {summary_res.content}"]
        self.msg_count = 0
        print("\n\n--- [System: History cleared and replaced with Summary] ---\n")
        # Print summary
        print("\n" + "*"*50)
        print("CONVERSATION SUMMARY:")
        print(summary_text)
        print("*"*50 + "\n")

    def chat(self):
        print("TravelGuide: Hello! I'm your travel expert. Where is your next adventure? Type 'exit' or 'quit' when done")

        while True:
            user_input = input("\nYou: ")
            if user_input.lower() in ["exit", "quit"]: break

            # Format the template with history and new input
            full_prompt = travel_prompt.format(
                history=self.history_text,
                input=user_input
            )

            print("TravelAssistant: ", end="", flush=True)
            full_reply = ""

            # Stream the response
            for chunk in llm.stream(full_prompt):
                content = chunk.content
                print(content, end="", flush=True)
                full_reply += content
            print("\n\n*** Type 'exit' or 'quit' when you are done ***")
            # Update history string and count
            self.history_text += f"\nHuman: {user_input}\nAI: {full_reply}\n"
            self.msg_count += 1

            # Summarize every 5 messages
            if self.msg_count >= 5:
                self.summarize_history()

if __name__ == "__main__":
    bot = TravelAssistantBot()
    bot.chat()

TravelGuide: Hello! I'm your travel expert. Where is your next adventure? Type 'exit' or 'quit' when done

You: What can you do?
TravelAssistant: I can assist with travel-related questions, such as suggesting destinations, planning itineraries, providing information on flights and accommodations, advising on travel tips and safety, and more! How can I help with your travel plans today?

*** Type 'exit' or 'quit' when you are done ***

You: What are top cuisines to try in Minneota?
TravelAssistant: I can't help with it.

*** Type 'exit' or 'quit' when you are done ***

You: Which places to visit in Minnesota?
TravelAssistant: Minnesota offers a variety of attractions and beautiful natural landscapes. Here are some top places to visit:

1. **Mall of America** - Located in Bloomington, it's the largest shopping mall in the United States, offering shopping, entertainment, and dining options.

2. **Boundary Waters Canoe Area Wilderness** - A stunning area known for its canoeing, fishing, an